In [ ]:
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel

model , tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-0.5B-Instruct",
    load_in_4bit = True,
    max_seq_length = 1024
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    lora_alpha = 16,
    target_modules = ["k_proj", "q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth"

)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files = "emails.jsonl")

In [ ]:
dataset

In [ ]:
def format(sample):
  formatted_dict = {
      "text" : tokenizer.apply_chat_template(sample["messages"], tokenize = False, add_generation_prompt = False)
  }

  return formatted_dict

In [ ]:
dataset = dataset.map(format, batched = False)

In [ ]:
dataset = dataset.remove_columns("messages")

In [ ]:
dataset["train"][0]

In [ ]:
!pip install trl


In [ ]:
from trl import SFTConfig

args = SFTConfig(
    output_dir = "./Emails LLM v4",
    num_train_epochs = 6,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    logging_steps = 15,
    save_strategy = "epoch",
    fp16 = True,
    report_to = "none",
    learning_rate = 2e-4
)

In [ ]:
dataset

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset['train'],
    args = args,
    max_seq_length = 1024
)

In [ ]:
trainer.train()

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
model.push_to_hub("vanshthadani/cs-support-v3")
tokenizer.push_to_hub("vanshthadani/cs-support-v3")

In [ ]:
email = "Hello, I received a Red Leather Wallet instead of the smartwatch I ordered. This is unacceptable. I want a refund."


prompt = [
    {
        "role" : "system", "content" : "You Classify Customer Support Emails into a structured JSON tickets"
    },
    {
        "role" : "user", "content" : email
    }
]

model.eval()

prompt = tokenizer.apply_chat_template(
    prompt,
    tokenize = True,
    add_generation_prompt = False,
    return_dict = True,
    return_tensors = "pt"
).to(model.device)

output = model.generate(
    **prompt,
    max_new_tokens = 200
)

response = tokenizer.decode(
    output[0][prompt["input_ids"].shape[1]:],
    skip_special_tokens = True
)
print(f"Email : {email}")

print(response)